# Feature table – step 1: metadata tables

This notebook builds the metadata part of the feature table: one CSV file per dataset and
subset (13 files), all with the same 13 columns in the same order. The next steps of the
feature table (silence, loudness, bandwidth, duration) start from these files.

**How to use it**

- Run the cells from top to bottom.
- The notebook only **reads** `datasets/`. It reads the label files
  and the lists of audio files, not the sound itself.

**Output files**

In `outputs/08_feature_table_output/metadata/` (not committed to git):

- `metadata_<dataset>_<subset>.csv`, e.g. `metadata_01_asvspoof_2019_eval.csv` (step 3).

**Steps**

1. Setup – paths
2. Read the metadata of all datasets
3. Save one CSV per dataset and subset

**Columns** (an empty cell = the dataset does not give this value or it does not apply)

| column | meaning |
|---|---|
| `dataset` | dataset, name of its folder in `datasets/` |
| `subset` | train / dev / eval (2019), progress / eval / hidden (2021), all (ITW), train / valid / test (SONICS) |
| `file_id` | file name without extension |
| `path` | path of the audio file relative to the project folder |
| `label` | `bonafide` or `spoof` |
| `attack_id` | system that made the fake (A01–A19, DF attacks, SONICS generator); `bonafide` for bona fide; empty when the dataset gives no attack (In-the-Wild spoof) |
| `speaker_id` | speaker (ASVspoof ID, ITW name); empty for SONICS |
| `codec` | codec condition applied before saving (2021 LA and DF only) |
| `transmission` | transmission channel (2021 LA only) |
| `source` | source corpus (2021 DF only): `asvspoof` = ASVspoof 2019 LA eval, `vcc2018`, `vcc2020` |
| `vocoder` | vocoder type (2021 DF only) |
| `trim` | `notrim`, or silence removed by the organisers in `hidden`: `only_speech` (LA), `trim` (DF) |
| `fake_type` | `full fake`, `half fake` or `mostly fake` (SONICS only) |

### Step 1 – Setup: paths

**What it does:** Loads the libraries, finds the project folder and sets the path to the
datasets (`datasets/`) and to the output folder (`outputs/08_feature_table_output/metadata`).

**Why:** To load the working data.

In [ ]:
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("pyproject.toml not found")


REPO_ROOT = find_repo_root(Path.cwd())
DATASETS_DIR = REPO_ROOT / "datasets"
NOTEBOOK_NAME = "08_feature_table"
OUTPUTS_DIR = REPO_ROOT / "outputs" / f"{NOTEBOOK_NAME}_output"
METADATA_DIR = OUTPUTS_DIR / "metadata"

print("Datasets folder:", DATASETS_DIR.relative_to(REPO_ROOT), "| exists:", DATASETS_DIR.exists())
print("Output folder:", METADATA_DIR.relative_to(REPO_ROOT))

### Step 2 – Read the metadata of all datasets

**What it does:** Calls the five readers from `src/xai_audio_deepfake_shortcuts_audit/metadata/`,
one per dataset. Each reader reads the label file of its dataset, finds the audio files on
disk, adds the path of every file and returns one table per subset (a dictionary
subset → table), all with the common columns. For every dataset it prints the number of rows in total and in each subset (13 subsets), and at
the end the number of rows in all datasets together.

**Why:** The metadata of every dataset is read so that it can be saved to CSV files in the
next step.

In [ ]:
from xai_audio_deepfake_shortcuts_audit.metadata.read_asvspoof_2019_metadata import (
    read_asvspoof_2019_metadata,
)
from xai_audio_deepfake_shortcuts_audit.metadata.read_asvspoof_2021_df_metadata import (
    read_asvspoof_2021_df_metadata,
)
from xai_audio_deepfake_shortcuts_audit.metadata.read_asvspoof_2021_la_metadata import (
    read_asvspoof_2021_la_metadata,
)
from xai_audio_deepfake_shortcuts_audit.metadata.read_in_the_wild_metadata import (
    read_in_the_wild_metadata,
)
from xai_audio_deepfake_shortcuts_audit.metadata.read_sonics_metadata import read_sonics_metadata

READERS = {
    "01_asvspoof_2019": read_asvspoof_2019_metadata,
    "02_asvspoof_2021_la": read_asvspoof_2021_la_metadata,
    "03_asvspoof_2021_df": read_asvspoof_2021_df_metadata,
    "04_in_the_wild": read_in_the_wild_metadata,
    "05_sonics": read_sonics_metadata,
}

metadata = {}
for dataset, read_metadata in READERS.items():
    tables = read_metadata(DATASETS_DIR)
    metadata[dataset] = tables
    print(f"{dataset}: {sum(len(table) for table in tables.values()):,} rows")
    for subset, table in tables.items():
        print(f"    {subset}: {len(table):,}")

total_rows = sum(len(table) for tables in metadata.values() for table in tables.values())
print(f"All datasets: {total_rows:,} rows")

### Step 3 – Save one CSV per dataset and subset

**What it does:** `save_subset_tables` writes the table of every subset to its own file
`metadata_<dataset>_<subset>.csv` in `outputs/08_feature_table_output/metadata/` and returns
the list of saved files with their number of rows. Files from an earlier run are overwritten.

**Why:** Every subset is saved to its own CSV file, so the files can be read separately or
joined later.

In [ ]:
import pandas as pd
from IPython.display import display

from xai_audio_deepfake_shortcuts_audit.metadata.save_subset_tables import save_subset_tables

saved = pd.concat(
    [save_subset_tables(tables, METADATA_DIR, "metadata") for tables in metadata.values()],
    ignore_index=True,
)
saved["path"] = [path.relative_to(REPO_ROOT).as_posix() for path in saved["path"]]
display(saved)